# 1-10 한국의 배당계급 — 몇 년째 배당을 늘려 왔나

**질문 예시:** "배당을 10년 넘게 계속 늘린 회사 있어?", "삼성전자·신한지주 배당 몇 년째 늘었어?", "배당 귀족 같은 거 한국에도 있어?"

미국에는 배당을 연속으로 늘린 햇수로 매기는 등급이 있습니다. 이 노트북은 같은 잣대를 국내 상장사에 적용합니다.

| 미국 등급 | 기준 | 이 노트북 |
|---|---|---|
| 배당 왕 (Dividend Kings) | 50년 이상 연속 증가 | 그대로 |
| 배당 귀족 (Aristocrats) | 25년 이상 + S&P500 편입 | **해당 없음** (지수 편입 조건이 국내에 없음) |
| 배당 챔피언 (Champions) | 25년 이상 | 그대로 |
| 배당 도전자 (Contenders) | 10~24년 | 그대로 |
| 배당 신규 진입자 (Challengers) | 5~9년 | 그대로 |

CHECK 원본(`1_기업펀더멘털/10_배당주`)은 배당수익률·배당성향·ROE로 몇 종목을 비교했습니다.
여기서는 **배당을 주는 전 종목**의 연속 증가 햇수를 계산하고 등급을 매깁니다. 원래 앱의 약점이던 종목별 반복 호출과 기준연도 하드코딩은 없앴습니다.

| 단계 | DeepSearch |
|---|---|
| 배당 지급 종목 | 자연어 질의 `… and (날짜 배당수익률 > 0)` 한 번 |
| 연도별 주당배당금(2013년~) | `GetDartDividends` (정기보고서 배당 표, 400종목씩 일괄) |
| 그 이전 이력(1996년~) | `GetCompanyDividends` (상위 후보만 일괄) |

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/1_fundamentals/10_dividend_class.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"   # 기준일 (배당수익률·배당 지급 여부를 보는 날)
EXTEND_TOP = 150      # 장기 이력으로 연장할 후보 최대 수

## 1. 배당을 주는 종목
기준일 배당수익률이 0보다 큰 코스피·코스닥 종목을 한 번에 받습니다. 우선주는 빼고 보통주만 남깁니다.

In [4]:
pay = ds_df(f"(('코스피' 상장 기업) or ('코스닥' 상장 기업)) and ({ASOF} 배당수익률 > 0)")
ycol = [c for c in pay.columns if c.startswith("배당수익률")][0]
u = universe()
pay = pay[["symbol", ycol]].rename(columns={ycol: "배당수익률(%)"}).merge(u, on="symbol", how="left")
pay = pay[pay["company_type_l1"].isin(["1", "2"]) & pay["symbol"].str.endswith("0")].reset_index(drop=True)
print(f"배당 지급 보통주 {len(pay):,}종목 (공기업 {int((pay['company_type_l1'] == '2').sum())}종목 포함)")
pay.groupby("market").size().rename("종목 수").to_frame()

배당 지급 보통주 1,249종목 (공기업 8종목 포함)


,종목 수
market,
KOSDAQ,679
KOSPI,570


## 2. 정기보고서 배당 표에서 보통주 주당배당금 뽑기
`GetDartDividends`는 사업·분기·반기보고서의 배당 표를 줍니다. 보고서마다 **당기·전기·전전기** 값이 함께 있어서,
2015년 사업보고서부터 받으면 2013 회계연도까지 거슬러 갑니다.

- **사업보고서만** 씁니다(보고서 기준일의 월 = 결산월).
- 주당 현금배당금 행 가운데 **우선주·종류주는 빼고**, `보통주`·`소액주주`·`의결권 있는 주식` 같은 표기를 보통주로 봅니다.
  차등배당(최대주주·소액주주 따로)이면 큰 값(대개 소액주주)을 씁니다. 표기가 비어 있으면 그 보고서에 다른 후보가 없을 때만 보통주로 봅니다.

In [5]:
import re
syms = pay["symbol"].tolist()
parts = [ds_df(f"GetDartDividends([{', '.join(syms[i:i + 400])}], date_from=2012-01-01, date_to={ASOF})")
         for i in range(0, len(syms), 400)]
dart = pd.concat(parts, ignore_index=True)
print(f"배당 표 {len(dart):,}행, {dart['symbol'].nunique():,}종목, 보고서 기준일 {dart['date'].min().date()} ~ {dart['date'].max().date()}")

PREF = re.compile(r"우선|종류|상환")
COMMON = re.compile(r"보통|보퉁|보동|일반주|소액주주|대주주|최대주주|의결권\s*있")
fye = u.set_index("symbol")["fiscal_year_end"].to_dict()

def common_dps(d):
    x = d[d["data_type"].str.replace(" ", "") == "주당현금배당금(원)"].copy()
    x["결산월"] = x["symbol"].map(fye).fillna(12).astype(int).replace(0, 12)
    x = x[x["date"].dt.month == x["결산월"]]                       # 사업보고서만
    x = x[~x["stock_type"].fillna("").str.contains(PREF)]           # 우선·종류주 제외
    x["표기"] = x["stock_type"].fillna("").str.contains(COMMON)

    def pick(g):
        e = g[g["표기"]]
        h = e if len(e) else g
        rule = "보통주 표기" if len(e) else ("표기 없음(행 1개)" if len(g) == 1 else "표기 없음(여러 행→최대)")
        return pd.Series({"this": h["this_period"].max(), "last": h["last_period"].max(),
                          "two": h["two_periods_ago"].max(), "규칙": rule})

    r = x.groupby(["symbol", "entity_name", "date"]).apply(pick, include_groups=False).reset_index()
    par = d[d["data_type"].str.replace(" ", "").str.startswith("주당액면가액")]
    par = par.groupby(["symbol", "date"])[["this_period", "last_period", "two_periods_ago"]].max()
    par.columns = ["par_this", "par_last", "par_two"]
    r = r.merge(par, left_on=["symbol", "date"], right_index=True, how="left")
    r["fy"] = r["date"].dt.year
    return r

rep = common_dps(dart)
print(f"사업보고서 {len(rep):,}건")
rep["규칙"].value_counts().rename("보고서 수").to_frame()

배당 표 256,987행, 1,248종목, 보고서 기준일 2014-12-31 ~ 2026-09-30


사업보고서 10,827건


,보고서 수
규칙,
보통주 표기,9521
표기 없음(행 1개),1264
표기 없음(여러 행→최대),42


## 3. 해마다 "늘렸나"를 판정하기 (액면분할 보정)
회계연도 Y의 증가 여부는 **Y 사업보고서 안의 당기 vs 전기**로 판정합니다. Y 보고서가 없으면 Y+1 보고서의 전기 vs 전전기를 씁니다.

같은 보고서 안이라도 **전기 배당을 분할 전 기준으로 적어 둔 회사가 있습니다**(삼성전자 2018 사업보고서: 당기 1,416원, 전기 42,500원).
그래서 같은 보고서의 **액면가(당기·전기)** 비율로 전기 배당을 보정합니다. 액면가와 배당을 함께 재작성한 회사는 비율이 1이라 영향이 없습니다.

- **연속배당증가**: 당기 > 전기 가 이어진 햇수 (아니면 0으로 리셋)
- **배당증가유지**: 당기 ≥ 전기 이고 당기 ≠ 0 이 이어진 햇수 (동결도 인정)

In [6]:
def yearly(r):
    rows = []
    for sym, g in r.groupby("symbol"):
        g = g.set_index("fy").sort_index()
        for y in range(g.index.min() - 1, g.index.max() + 1):
            if y in g.index:
                cur, prev, pa, pb, src = g.at[y, "this"], g.at[y, "last"], g.at[y, "par_this"], g.at[y, "par_last"], f"{y} 보고서"
            elif y + 1 in g.index:
                cur, prev, pa, pb, src = g.at[y + 1, "last"], g.at[y + 1, "two"], g.at[y + 1, "par_last"], g.at[y + 1, "par_two"], f"{y + 1} 보고서(전기)"
            else:
                continue
            adj = pa / pb if (pd.notna(pa) and pd.notna(pb) and pa > 0 and pb > 0 and pa != pb) else 1.0
            if adj != 1.0:
                src += f", 액면 {pb:,.0f}→{pa:,.0f} 보정"
            cur = 0.0 if pd.isna(cur) else float(cur)
            prev = None if pd.isna(prev) else float(prev) * adj
            rows.append({"symbol": sym, "fy": y, "dps": cur, "전기(보정)": prev, "출처": src,
                         "증가": None if prev is None else cur > prev,
                         "증가·유지": None if prev is None else (cur >= prev and cur != 0)})
    return pd.DataFrame(rows)

yr = yearly(rep)
END = int(rep.groupby("symbol")["fy"].max().value_counts().idxmax())   # 가장 많은 종목이 사업보고서를 낸 최신 연도
first_fy = yr.dropna(subset=["증가"]).groupby("symbol")["fy"].min()     # 종목별 첫 판정 연도
FIRST = int(first_fy.value_counts().idxmax())
OBS_MAX = END - FIRST + 1
print(f"기준 회계연도 {END} (최신 완결 연도 자동 판정) · 대부분 종목의 첫 판정 연도 {FIRST} "
      f"→ DART만으로 관측 가능한 최대 연속 {OBS_MAX}년")

def run_length(flags, end):
    n, y = 0, end
    while flags.get(y) is True:
        n, y = n + 1, y - 1
    return n

st = []
for sym, g in yr.groupby("symbol"):
    inc, keep = dict(zip(g["fy"], g["증가"])), dict(zip(g["fy"], g["증가·유지"]))
    st.append({"symbol": sym, "연속배당증가": run_length(inc, END), "배당증가유지": run_length(keep, END),
               "END 보고서": END in set(g["fy"])})
st = pd.DataFrame(st)
st["첫 판정 연도"] = st["symbol"].map(first_fy)
st["관측 한계"] = END - st["첫 판정 연도"] + 1          # 이 종목을 DART로 셀 수 있는 최대 연수
dist =pd.DataFrame({"연속배당증가": st["연속배당증가"].value_counts(), "배당증가유지": st["배당증가유지"].value_counts()}).fillna(0).astype(int).sort_index()
dist.index.name = "연수"
dist.T

기준 회계연도 2025 (최신 완결 연도 자동 판정) · 대부분 종목의 첫 판정 연도 2014 → DART만으로 관측 가능한 최대 연속 12년


연수,0,1,2,3,4,5,6,7,8,9,10,11,12
연속배당증가,672,288,140,62,36,15,12,4,3,2,1,0,2
배당증가유지,322,188,178,127,68,63,51,42,31,19,13,17,118


## 4. 관측 한계에 닿은 종목은 더 오래된 이력으로 연장
DART 표만으로는 최대 위 연수까지만 셀 수 있습니다. 그 한계에 닿은 종목은 `GetCompanyDividends`(1996년~)로 이어 붙입니다.

이 데이터는 **분할 보정이 안 된 원래 값**이고, 실제로는 배당했는데 **0원으로 들어 있는 해**가 있습니다(예: 신한지주 2023년 0원, DART에는 2,100원).
그래서 다음 규칙으로 보수적으로 셉니다.
- DART와 겹치는 해는 DART 값을 씁니다. 연장은 그 종목의 DART 첫 판정 연도 이전으로만 합니다.
- 이어 붙이는 첫해는 두 출처의 주당배당금이 같아야 합니다(다르면 "출처 불일치"로 멈춤).
- 직전 해의 30% 미만으로 급감하면 **분할인지 실제 감배(예: 1997~98년 외환위기)인지 구분할 수 없어** 멈추고, 앞뒤 해가 양수인데 가운데가 0원이면 **결측 가능**으로 보고 거기서 멈춥니다(연장하지 않음).

In [7]:
hit = ((st["연속배당증가"] >= st["관측 한계"]) | (st["배당증가유지"] >= st["관측 한계"])) & (st["관측 한계"] >= 5)
cand = st[hit].sort_values("관측 한계", ascending=False)["symbol"].tolist()[:EXTEND_TOP]
print(f"DART 관측 한계(5년 이상)에 닿은 종목 {int(hit.sum())}개 중 {len(cand)}개를 연장 후보로")
gcd = ds_df(f"GetCompanyDividends([{', '.join(cand)}], date_from=1990-01-01, date_to={ASOF})") if cand else pd.DataFrame()
DPS_COLS = ["주당현금배당액(소주주,보통주)(원)", "보통주현금배당액"]

def gcd_series(sym):
    a = gcd[(gcd["symbol"] == sym) & (gcd["accounting_type"] == "1") & gcd["dividend_type_name"].isin(DPS_COLS)]
    if a.empty:
        return pd.Series(dtype=float)
    p = a.pivot_table(index="date", columns="dividend_type_name", values="amount", aggfunc="max")
    s = p[DPS_COLS[0]] if DPS_COLS[0] in p else p[DPS_COLS[1]]
    if DPS_COLS[1] in p:
        s = s.fillna(p[DPS_COLS[1]])
    s.index = s.index.year
    return s.groupby(level=0).max().sort_index()   # 같은 해에 연간 행이 둘이면 큰 값

def extend(sym, key, base):
    """DART로 관측 한계까지 이어졌으면, 그 종목의 첫 판정 연도 직전 해부터 거꾸로 GetCompanyDividends로 이어 센다."""
    row = st[st["symbol"] == sym].iloc[0]
    if sym not in cand or base < row["관측 한계"]:
        return base, ""
    s = gcd_series(sym)
    f0 = yr[(yr["symbol"] == sym) & (yr["fy"] == row["첫 판정 연도"])]
    dart_y0 = f0["전기(보정)"].iloc[0] if len(f0) else None     # DART가 아는 가장 오래된 해의 주당배당금
    y = int(row["첫 판정 연도"]) - 1
    g0 = s.get(y)
    if dart_y0 is None or g0 is None or pd.isna(g0) or abs(g0 - dart_y0) > 0.5:
        return base, f"{y}년 출처 불일치(DART {dart_y0}, GCD {g0})로 연장 안 함"
    n, note = base, ""
    while True:
        cur, prev = s.get(y), s.get(y - 1)
        if cur is None or prev is None or pd.isna(cur) or pd.isna(prev):
            note = f"{y - 1}년 이전 자료 없음"
            break
        nxt = s.get(y - 2)
        if prev == 0 and cur > 0 and nxt is not None and nxt > 0:
            note = f"{y - 1}년 0원(결측 가능)에서 멈춤"
            break
        if prev > 0 and cur < 0.3 * prev:
            note = f"{y}년 급감(분할 또는 실제 감배)에서 멈춤"
            break
        ok = cur > prev if key == "연속배당증가" else (cur >= prev and cur != 0)
        if not ok:
            note = f"{y}년에 끊김"
            break
        n, y = n + 1, y - 1
    return n, note

for key in ("연속배당증가", "배당증가유지"):
    res = [extend(s_, key, b) for s_, b in zip(st["symbol"], st[key])]
    st[key + "(연장)"] = [r[0] for r in res]
    st[key + " 메모"] = [r[1] for r in res]
ext = st[st["symbol"].isin(cand)].merge(u[["symbol", "name"]], on="symbol")
print("연장으로 늘어난 종목 수 — 연속배당증가:", int((ext["연속배당증가(연장)"] > ext["연속배당증가"]).sum()),
      "/ 배당증가유지:", int((ext["배당증가유지(연장)"] > ext["배당증가유지"]).sum()))
print("멈춘 이유(배당증가유지):", ext["배당증가유지 메모"].str.replace(r"^\d{4}년\s*", "", regex=True)
      .str.replace(r"\(DART.*", "", regex=True).value_counts().to_dict())
ext[["name", "연속배당증가", "연속배당증가(연장)", "연속배당증가 메모", "배당증가유지", "배당증가유지(연장)", "배당증가유지 메모"]] \
    .sort_values(["배당증가유지(연장)", "연속배당증가(연장)"], ascending=False).head(15).reset_index(drop=True)

DART 관측 한계(5년 이상)에 닿은 종목 156개 중 150개를 연장 후보로


연장으로 늘어난 종목 수 — 연속배당증가: 0 / 배당증가유지: 100
멈춘 이유(배당증가유지): {'에 끊김': 81, '이전 자료 없음': 26, '급감(분할 또는 실제 감배)에서 멈춤': 20, '출처 불일치': 13, '0원(결측 가능)에서 멈춤': 10}


,name,연속배당증가,연속배당증가(연장),연속배당증가 메모,배당증가유지,배당증가유지(연장),배당증가유지 메모
0,LS,4,4,,12,28,1997년 급감(분할 또는 실제 감배)에서 멈춤
1,농심,1,1,,12,28,1997년 급감(분할 또는 실제 감배)에서 멈춤
2,SK가스,1,1,,12,28,1997년 급감(분할 또는 실제 감배)에서 멈춤
3,태경비케이,0,0,,12,28,1997년에 끊김
4,KISCO홀딩스,6,6,,12,27,1998년에 끊김
5,해성산업,1,1,,12,26,1999년 급감(분할 또는 실제 감배)에서 멈춤
6,평화홀딩스,0,0,,12,26,1999년 급감(분할 또는 실제 감배)에서 멈춤
7,하이록코리아,4,4,,12,25,2000년 급감(분할 또는 실제 감배)에서 멈춤
8,유한양행,3,3,,12,25,2000년에 끊김
9,와이어블,0,0,,12,25,2000년에 끊김


## 5. 배당계급표
연장까지 반영한 **연속배당증가** 햇수로 등급을 매기고, 최근 5년 주당배당금, 배당수익률, 배당성향(현금배당총액 ÷ 연결 당기순이익)을 붙입니다.
동결을 인정하는 **배당증가유지** 기준 등급도 옆에 둡니다(국내는 배당을 몇 년 동결하는 경우가 많아 이쪽이 실감에 가깝습니다).

In [8]:
def tier(n):
    if n >= 50: return "👑 배당 왕 (50년+)"
    if n >= 25: return "🏆 배당 챔피언 (25년+)"
    if n >= 10: return "🏅 배당 도전자 (10~24년)"
    if n >= 5:  return "🚀 신규 진입자 (5~9년)"
    return "-"

def recent5(sym):
    g = yr[yr["symbol"] == sym].set_index("fy")["dps"]
    return " → ".join(f"{g.get(y, float('nan')):,.0f}" for y in range(END - 4, END + 1))

fy_end = dart[(dart["date"].dt.year == END) & (dart["date"].dt.month == dart["symbol"].map(fye).fillna(12).astype(int).replace(0, 12))]
tot = fy_end[fy_end["data_type"].str.contains("현금배당금총액")].groupby("symbol")["this_period"].max()
ni = None
for label in ("(연결)당기순이익", "(별도)당기순이익", "(개별)당기순이익"):   # 연결이 없으면 별도·개별
    s_ = fy_end[fy_end["data_type"].str.startswith(label)].groupby("symbol")["this_period"].max()
    ni = s_ if ni is None else ni.combine_first(s_)
payout = (tot / ni.where(ni > 0) * 100).rename("배당성향(%)")   # 순이익 0 이하(적자)면 비움

board = st.merge(pay[["symbol", "name", "market", "배당수익률(%)"]], on="symbol").merge(payout, left_on="symbol", right_index=True, how="left")
board["등급(증가)"] = board["연속배당증가(연장)"].map(tier)
board["등급(증가·유지)"] = board["배당증가유지(연장)"].map(tier)
board[f"DPS {END - 4}~{END}"] = board["symbol"].map(recent5)
board = board.sort_values(["연속배당증가(연장)", "배당증가유지(연장)", "배당수익률(%)"], ascending=False)

ORDER = ["👑 배당 왕 (50년+)", "🏆 배당 챔피언 (25년+)", "🏅 배당 도전자 (10~24년)", "🚀 신규 진입자 (5~9년)", "-"]
summary = pd.DataFrame({"연속배당증가 기준": board["등급(증가)"].value_counts(),
                        "배당증가유지 기준": board["등급(증가·유지)"].value_counts()}).reindex(ORDER).fillna(0).astype(int)
summary.rename(index={"-": "5년 미만"})

,연속배당증가 기준,배당증가유지 기준
👑 배당 왕 (50년+),0,0
🏆 배당 챔피언 (25년+),0,10
🏅 배당 도전자 (10~24년),3,139
🚀 신규 진입자 (5~9년),36,205
5년 미만,1198,883


In [9]:
cols = ["name", "market", "연속배당증가(연장)", "배당증가유지(연장)", "등급(증가)", f"DPS {END - 4}~{END}", "배당수익률(%)", "배당성향(%)"]
top = board[board["연속배당증가(연장)"] >= 5][cols].round(2).reset_index(drop=True)
top.index += 1
top.head(30)

,name,market,연속배당증가(연장),배당증가유지(연장),등급(증가),DPS 2021~2025,배당수익률(%),배당성향(%)
1,고려신용정보,KOSDAQ,12,17,🏅 배당 도전자 (10~24년),300 → 330 → 380 → 450 → 500,6.16,53.23
2,NICE평가정보,KOSPI,12,12,🏅 배당 도전자 (10~24년),330 → 370 → 410 → 460 → 510,3.15,38.21
3,경동나비엔,KOSPI,10,14,🏅 배당 도전자 (10~24년),450 → 500 → 550 → 650 → 750,1.28,12.08
4,영원무역홀딩스,KOSPI,9,20,🚀 신규 진입자 (5~9년),"2,000 → 3,050 → 3,970 → 5,350 → 6,576",4.29,20.76
5,JB금융지주,KOSPI,9,10,🚀 신규 진입자 (5~9년),"599 → 835 → 855 → 995 → 1,140",4.62,30.02
6,한국타이어앤테크놀로지,KOSPI,8,13,🚀 신규 진입자 (5~9년),"700 → 800 → 1,300 → 2,000 → 2,300",3.93,25.75
7,NICE,KOSPI,8,12,🚀 신규 진입자 (5~9년),375 → 410 → 451 → 500 → 560,4.99,52.00
8,클래시스,KOSDAQ,8,8,🚀 신규 진입자 (5~9년),"66 → 116 → 200 → 257 → 1,000",4.93,49.38
9,넥센,KOSPI,7,19,🚀 신규 진입자 (5~9년),100 → 110 → 125 → 135 → 175,3.04,5.12
10,한솔로지스틱스,KOSPI,7,17,🚀 신규 진입자 (5~9년),60 → 70 → 100 → 150 → 200,7.74,35.54


### 동결을 인정하면 (배당증가유지 10년 이상, 배당수익률 순)

In [10]:
keep10 = board[board["배당증가유지(연장)"] >= 10].sort_values("배당수익률(%)", ascending=False)[cols].round(2).reset_index(drop=True)
keep10.index += 1
print(f"배당증가유지 10년 이상: {len(keep10)}종목")
keep10.head(20)

배당증가유지 10년 이상: 149종목


,name,market,연속배당증가(연장),배당증가유지(연장),등급(증가),DPS 2021~2025,배당수익률(%),배당성향(%)
1,와토스코리아,KOSDAQ,1,11,-,200 → 200 → 200 → 200 → 403,12.92,NaN
2,한세예스24홀딩스,KOSPI,1,19,-,250 → 250 → 250 → 250 → 500,12.36,NaN
3,화승인더,KOSPI,2,11,-,188 → 188 → 188 → 200 → 206,11.43,51.04
4,SOOP,KOSDAQ,3,13,-,"730 → 730 → 850 → 1,500 → 3,380",10.20,35.37
5,예스24,KOSDAQ,1,18,-,200 → 200 → 200 → 200 → 250,9.82,NaN
6,대덕,KOSPI,2,17,-,"300 → 400 → 400 → 500 → 1,155",9.55,1059.94
7,DSR제강,KOSPI,2,22,-,40 → 60 → 60 → 100 → 430,9.52,25.10
8,정상제이엘에스,KOSDAQ,0,17,-,530 → 530 → 530 → 530 → 530,9.40,108.39
9,HS애드,KOSPI,0,15,-,350 → 400 → 450 → 550 → 550,9.02,51.30
10,제일연마,KOSPI,3,10,-,270 → 270 → 300 → 320 → 721,8.79,35.11


## 6. 손으로 검산
몇 종목은 연도별 판정을 그대로 펼쳐 봅니다. 출처 열에 어느 보고서에서 읽었는지, 액면 보정이 들어갔는지 나옵니다.

In [11]:
CHECKS = {"KRX:055550": "신한지주", "KRX:005930": "삼성전자", "KRX:000270": "기아", "KRX:033780": "KT&G"}
for sym, nm in CHECKS.items():
    g = yr[yr["symbol"] == sym][["fy", "dps", "전기(보정)", "증가", "증가·유지", "출처"]].tail(8)
    b = board[board["symbol"] == sym]
    head = f"{nm}: 연속배당증가 {int(b['연속배당증가(연장)'].iloc[0])}년, 배당증가유지 {int(b['배당증가유지(연장)'].iloc[0])}년" if len(b) else nm
    print(head)
    display(g.reset_index(drop=True))

신한지주: 연속배당증가 5년, 배당증가유지 5년


,fy,dps,전기(보정),증가,증가·유지,출처
0,2018,1600.0,1450.0,True,True,2018 보고서
1,2019,1850.0,1600.0,True,True,2019 보고서
2,2020,1500.0,1850.0,False,False,2020 보고서
3,2021,1960.0,1500.0,True,True,2021 보고서
4,2022,2065.0,1960.0,True,True,2022 보고서
5,2023,2100.0,2065.0,True,True,2023 보고서
6,2024,2160.0,2100.0,True,True,2024 보고서
7,2025,2590.0,2160.0,True,True,2025 보고서


삼성전자: 연속배당증가 2년, 배당증가유지 4년


,fy,dps,전기(보정),증가,증가·유지,출처
0,2018,1416.0,850.0,True,True,"2018 보고서, 액면 5,000→100 보정"
1,2019,1416.0,1416.0,False,True,2019 보고서
2,2020,2994.0,1416.0,True,True,2020 보고서
3,2021,1444.0,2994.0,False,False,2021 보고서
4,2022,1444.0,1444.0,False,True,2022 보고서
5,2023,1444.0,1444.0,False,True,2024 보고서(전기)
6,2024,1446.0,1444.0,True,True,2024 보고서
7,2025,1668.0,1446.0,True,True,2025 보고서


기아: 연속배당증가 5년, 배당증가유지 5년


,fy,dps,전기(보정),증가,증가·유지,출처
0,2018,900.0,800.0,True,True,2018 보고서
1,2019,1150.0,900.0,True,True,2019 보고서
2,2020,1000.0,1150.0,False,False,2020 보고서
3,2021,3000.0,1000.0,True,True,2021 보고서
4,2022,3500.0,3000.0,True,True,2022 보고서
5,2023,5600.0,3500.0,True,True,2023 보고서
6,2024,6500.0,5600.0,True,True,2024 보고서
7,2025,6800.0,6500.0,True,True,2025 보고서


KT&G: 연속배당증가 4년, 배당증가유지 17년


,fy,dps,전기(보정),증가,증가·유지,출처
0,2018,4000.0,4000.0,False,True,2018 보고서
1,2019,4400.0,4000.0,True,True,2019 보고서
2,2020,4800.0,4400.0,True,True,2020 보고서
3,2021,4800.0,4800.0,False,True,2021 보고서
4,2022,5000.0,4800.0,True,True,2022 보고서
5,2023,5200.0,5000.0,True,True,2023 보고서
6,2024,5400.0,5200.0,True,True,2024 보고서
7,2025,6000.0,5400.0,True,True,2025 보고서


In [12]:
# GetCompanyDividends 와 DART 가 어긋나는 해 (DART 쪽이 맞는 사례 확인용)
chk = ds_df(f"GetCompanyDividends([{', '.join(CHECKS)}], date_from=2013-01-01, date_to={ASOF})")
cmp_rows = []
for sym, nm in CHECKS.items():
    s = chk[(chk["symbol"] == sym) & (chk["accounting_type"] == "1") & (chk["dividend_type_name"] == DPS_COLS[0])]
    s = s.set_index(s["date"].dt.year)["amount"]
    for _, r in yr[yr["symbol"] == sym].iterrows():
        g = s.get(r["fy"])
        if g is not None and abs(g - r["dps"]) > 0.5:
            cmp_rows.append({"종목": nm, "회계연도": r["fy"], "DART": r["dps"], "GetCompanyDividends": g})
pd.DataFrame(cmp_rows) if cmp_rows else "두 출처가 모두 일치"

,종목,회계연도,DART,GetCompanyDividends
0,신한지주,2023,2100.0,0.0


## 결과 읽는 법과 한계
- **국내에는 25년 이상 연속 증가(챔피언)가 사실상 없습니다.** 연속 증가는 10년 안팎이 최상위이고, 동결을 인정하면 훨씬 많은 종목이 10년 이상 이어집니다.
- **배당 귀족**은 S&P500 편입 조건이라 국내에 대응 개념이 없어 매기지 않았습니다.
- DART 배당 표로는 **2013 회계연도부터** 판정됩니다. 그 이전은 `GetCompanyDividends`로 연장했는데, 분할 보정이 없고 0원 결측이 섞여 있어 **급감·0원에서 멈추는 보수적 기준**입니다(실제 연속 햇수보다 짧게 나올 수 있음).
- 액면분할은 보정하지만 **무상증자로 주식 수가 늘어난 해**는 보정하지 않습니다. 이때 주당배당금이 줄어 연속이 끊긴 것처럼 보일 수 있습니다.
- 중간·분기배당은 사업보고서의 연간 합계 기준입니다. 특별배당을 준 다음 해는 "감소"로 셉니다(삼성전자 2020→2021).
- 배당 지급 종목은 **기준일에 배당수익률이 있는 현재 상장사**입니다. 배당을 끊은 회사, 상장폐지된 회사는 빠집니다.